# 02 - Limpieza avanzada de embargos OCR

Objetivo: aplicar una segunda limpieza conservadora sobre `embargos_limpios.csv`, enfocada en ruido OCR y legibilidad, sin sobrescribir archivos originales ni completar datos legales que no estén claramente presentes.

Reglas de trabajo:

- Se trabaja únicamente con embargos.
- La columna base se conserva como `texto_base`.
- El resultado nuevo se guarda como `texto_limpieza_avanzada`.
- Los registros irrecuperables se marcan, no se eliminan.
- Ante la duda, se conserva texto para no perder expediente, DNI, CUIL/CUIT, CBU/CVU, montos, fechas, nombres, juzgados o carátulas.

# 1. Imports y configuración inicial

In [1]:
from pathlib import Path
import difflib
import json
import re
import unicodedata

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = PROCESSED_DIR / "embargos_limpios.csv"
OUTPUT_PATH = PROCESSED_DIR / "embargos_limpieza_avanzada.csv"
TOP15_PATH = PROCESSED_DIR / "embargos_15_mejores_legibles.csv"
REPORT_DIR = PROCESSED_DIR / "reportes_embargos"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

print("Proyecto:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Output principal:", OUTPUT_PATH)
print("Output top 15:", TOP15_PATH)
print("Reportes:", REPORT_DIR)

Proyecto: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia
Input: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_limpios.csv
Output principal: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_limpieza_avanzada.csv
Output top 15: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_15_mejores_legibles.csv
Reportes: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\reportes_embargos


# 2. Carga del dataset

En esta etapa solo se diagnostica el archivo de entrada. No se modifica el texto todavía.

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo de entrada: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)

print("Filas, columnas:", df.shape)
print("Columnas disponibles:")
for col in df.columns:
    print(f"- {col} ({df[col].dtype})")

print("\nNulos por columna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("nulos"))

if "clasificacion" in df.columns:
    print("\nValores en clasificacion:")
    display(df["clasificacion"].value_counts(dropna=False).to_frame("cantidad"))
    no_embargos = df[~df["clasificacion"].astype(str).str.contains("embargo", case=False, na=False)]
    if len(no_embargos) > 0:
        print(f"ATENCION: hay {len(no_embargos)} registros cuya clasificacion no contiene 'embargo'. No se eliminan; revisar antes de usar resultados.")

Filas, columnas: (627, 14)
Columnas disponibles:
- id (int64)
- texto_ocr (str)
- nombre (str)
- clasificacion (str)
- tiene_html_detectado (bool)
- patrones_html_detectados (str)
- estado_texto_ocr (str)
- largo_texto_ocr (int64)
- largo_original (int64)
- texto_limpio (str)
- texto_normalizado_busqueda (str)
- largo_limpio (int64)
- texto_markdown (str)
- diferencia_largo (int64)

Nulos por columna:


,nulos
id,0
texto_ocr,0
nombre,0
clasificacion,0
tiene_html_detectado,0
patrones_html_detectados,0
estado_texto_ocr,0
largo_texto_ocr,0
largo_original,0
texto_limpio,0



Valores en clasificacion:


,cantidad
clasificacion,
Embargo,627


In [3]:
CANDIDATAS_TEXTO = [
    "texto_limpio",
    "texto_markdown",
    "texto_ocr",
    "texto_original",
    "descripcion",
    "Description",
    "body",
    "texto",
]

columnas_texto = []
for col in df.columns:
    if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
        largos = df[col].fillna("").astype(str).str.len()
        columnas_texto.append({
            "columna": col,
            "no_nulos": int(df[col].notna().sum()),
            "largo_promedio": float(largos.mean()),
            "largo_mediana": float(largos.median()),
            "largo_max": int(largos.max()),
        })

columnas_texto_df = pd.DataFrame(columnas_texto).sort_values("largo_promedio", ascending=False)
display(columnas_texto_df)

texto_col = next((c for c in CANDIDATAS_TEXTO if c in df.columns), None)
if texto_col is None:
    texto_col = columnas_texto_df.iloc[0]["columna"]

print(f"Columna seleccionada como texto base para limpieza avanzada: {texto_col}")

,columna,no_nulos,largo_promedio,largo_mediana,largo_max
7,texto_markdown,627,3613.885167,3409.0,11043
0,texto_ocr,627,3520.240829,3306.0,10930
5,texto_limpio,627,3496.878788,3292.0,10926
6,texto_normalizado_busqueda,627,3469.594896,3279.0,10805
1,nombre,627,17.006380,17.0,18
2,clasificacion,627,7.000000,7.0,7
3,patrones_html_detectados,627,2.012759,2.0,6
4,estado_texto_ocr,627,2.000000,2.0,2


Columna seleccionada como texto base para limpieza avanzada: texto_limpio


In [4]:
texto_base_tmp = df[texto_col].fillna("").astype(str)
longitudes = texto_base_tmp.str.len()
palabras = texto_base_tmp.str.findall(r"\b\w+\b", flags=re.UNICODE).str.len()

caracteres_raros_por_texto = texto_base_tmp.apply(
    lambda t: sum(
        1
        for ch in t
        if (not ch.isprintable() and ch not in "\n\r\t")
        or unicodedata.category(ch).startswith("C")
    )
)

resumen_diagnostico = pd.DataFrame({
    "metrica": [
        "filas_totales",
        "textos_vacios",
        "textos_muy_cortos_menos_100_chars",
        "longitud_minima",
        "longitud_maxima",
        "longitud_promedio",
        "longitud_mediana",
        "palabras_promedio",
        "caracteres_raros_total",
    ],
    "valor": [
        len(df),
        int((texto_base_tmp.str.strip() == "").sum()),
        int((longitudes < 100).sum()),
        int(longitudes.min()),
        int(longitudes.max()),
        round(float(longitudes.mean()), 2),
        round(float(longitudes.median()), 2),
        round(float(palabras.mean()), 2),
        int(caracteres_raros_por_texto.sum()),
    ],
})

display(resumen_diagnostico)
resumen_diagnostico.to_csv(REPORT_DIR / "diagnostico_inicial_resumen.csv", index=False, encoding="utf-8-sig")

print("Ejemplos aleatorios:")
display(df.sample(min(5, len(df)), random_state=RANDOM_STATE)[[c for c in ["id", "nombre", "clasificacion", texto_col] if c in df.columns]])

print("Textos mas cortos:")
display(df.assign(_largo=longitudes).sort_values("_largo").head(5)[[c for c in ["id", "nombre", "clasificacion", "_largo", texto_col] if c in df.columns or c == "_largo"]])

print("Textos mas largos:")
display(df.assign(_largo=longitudes).sort_values("_largo", ascending=False).head(5)[[c for c in ["id", "nombre", "clasificacion", "_largo", texto_col] if c in df.columns or c == "_largo"]])

,metrica,valor
0,filas_totales,627.00
1,textos_vacios,0.00
2,textos_muy_cortos_menos_100_chars,0.00
3,longitud_minima,285.00
4,longitud_maxima,10926.00
5,longitud_promedio,3496.88
6,longitud_mediana,3292.00
7,palabras_promedio,594.41
8,caracteres_raros_total,64323.00


Ejemplos aleatorios:


,id,nombre,clasificacion,texto_limpio
580,544832,Embargo - usuario,Embargo,"OFICIO JUDICIAL\n\n/ L/)\n\nBuenos Aires, de Febrero/de/2026\nAl Sr. Apoderado de |\n""MERCADO LIBRE S.R.L.""\nS / D\n\nDe mi consideración:\n\nTengo el agrado de dirigirme a Ud. en los autos\ncaratulados ""CHUQUISACA, ..."
590,546230,Embargo - usuario,Embargo,"A\n\n+ 19 MAR 2026 |\n\n+\n1\nÉ\n\nRECEPCION\n\ne\nPoder Judicial de la Nación\nJUZGADO CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFI..."
550,543050,Embargo - usuario,Embargo,TEXTO Y DATOS DE LA NOTIFICACIÓN\n\n[\n| Usuario conectado:\n\nMINGIACA GABRIEL ALEJANDRO - 202610119308 nofificaciones.scba.gov.ar\n\n| Organismo: JUZGADO DE FAMILIA N? 5 - SAN ISIDRO\n| Carátula: MUGUILLO GERVASIO ...
213,518820,Embargo - usuario,Embargo,"a (la\n, CART, . DOCUMENTO\nDESTINATARIO\nPrimera Instancia\nComercial N* 9 MERCADO LIBRE S.R.L.\n13\nDOMICILIO\n1 == >= — yda. € s N* 3039, Piso 2\nEz X= =WX= | LOCALIDAD PROVINGA\npe == =.1= 264A4 Ciudad Autónoma\n..."
485,538081,Embargo - usuario,Embargo,8 UN 00000\n100009755236\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL N*10 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MA...


Textos mas cortos:


,id,nombre,clasificacion,_largo,texto_limpio
509,542136,Embargo - usuario,Embargo,285,"q a ZA mercad\nA Ó-—\ntArñ20190D 198\n\nPOMADA\n\n40 007. vol sh oben Wisnewslal ssobañb"" Wbiscineua y lorinoo lo\n\nBm 3 me DATE bs D ADA\n\nln MOO .1 7 alansoro.o\ndm y o PrRMOONS 02 |\n\nr ' As\n1\n4 |\na\n1\ny\n!..."
329,526706,Embargo - usuario,Embargo,627,"OFICIO DE LEVANTAMIENTO DE EMBARGO\n\nCaba, de Febrero di\n\nctor de Mercado Libre SRL\n\n¡\n\nel agrado de dirigirme a Ud en el marco de los autos caratulados ""FERNANDEZ, JORGE\n\nL c/ COOPERATIVA DE TRABAJO LOGI MA..."
614,547367,Embargo - usuario,Embargo,706,"Sy\n\nCERTIFICA que la presente es copia fel cuyo original\nsido firmado digalmente por: ÉRUNELLO, Analia -\n\nPROSECRETARIO/A LETRADO. FASSETTA,\n\nDi Ignacio - JUE,\n\nÑ 5\nÑ 2\n\n. GOACO,\n\nFi di e,\nEM reto guin..."
102,508916,Embargo - usuario,Embargo,711,"Podor AÁadicial, de la O Maid\n\nOFICIO JUDICIAL\n\nBuenos Aires, 23 de Diciériibre de 2025\n\nque tramitan por ante este Juzgado de Primera Instancia del Ñl vajo N 22;\nmi cargo en Carácter de Juez subrogante. Secre..."
187,515755,Embargo - usuario,Embargo,749,"PSN\n| FS\nEl E\n| A o . )\n€ Ai\no. ""z ULA\nPoder Judicial de la Nación |\n| Juzgado Nacional de 1* Instancia del Trabajo N* 48\nl OFICIO JUDICIAL\n¡ Buenos Aires,03 de no de 2026\nAl Sh Director de Asuntos Legales\..."


Textos mas largos:


,id,nombre,clasificacion,_largo,texto_limpio
35,496830,Embargo - usuario,Embargo,10926,UPREMA DE JUSTICIA Fecha Impresión\n| 26/12/2025 - 08:53:52\n\n1125. | A ecoroasrass MN TAR\n\nDEN 0\n\n| JEICIO CASILLERO VIRTUAL CON FIRMA DIGITAL\n\nito: [2/12/2025 00:00\nbsitada en el/los domicilio/s digitales:\...
475,538037,Embargo - usuario,Embargo,9437,qua pr po ZEXTO y DA D3 DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRONICAS EF)\nX\n\n¿Usuario conectado: THEILER Pablo Maximiliano - 20322215232(Qnotificaciones.scba.gov.ar\n\nOrganismo: JUZ...
168,514558,Embargo - usuario,Embargo,9372,"23/12/25, Lie p.m. TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES E\nTEXTO Y DATOS DE LA NOTIFICACIÓN 9235 (331\nUsuario con d tado: SANTONOCITO MARTIN EZEQUIEL - 20261465311 (Qnotifica..."
66,502593,Embargo - usuario,Embargo,9085,"OFICIO JUDICIAL\nMANDATARIA JUDICIAL N* 33 |\n21 de enerp de 2026\npe\n\nBuenos Aires,\n\nesidente\nLibre SRL\n\nUsted en los autos caratulados ""GCBA\n¡ONAL SA EJECUCION FISCAL(EXPTE 318062/2024) que tramita pot ante..."
4,494151,Embargo - usuario,Embargo,8875,SDE LA NOTIFICACIÓN (\nFRAPPA LETICIA MABEL - 27295403603Qnotificaciones.scba.gov.ar |\nJUZGADO EN LO CIVIL Y COMERCIAL N? 1 - LA MATANZA |\nFRAPPA LETICIA MABEL C/ VALENTIN! JOSE LUIS S/ EJECUCION DE MEA\n\nM-3154-2...


# 3. Diagnóstico de calidad OCR

Se crean métricas trazables y un `ocr_quality_score` entre 0 y 1. El score no decide eliminación de registros; solo ayuda a priorizar revisión y selección posterior.

In [5]:
LEGAL_TERMS = [
    "embargo", "embargado", "expediente", "expte", "autos", "caratula", "carátula",
    "juzgado", "secretaria", "secretaría", "demandado", "demandante", "banco",
    "transferencia", "cuenta", "cbu", "cvu", "cuil", "cuit", "dni", "monto",
    "pesos", "sentencia", "medida cautelar", "oficio", "tribunal", "juez", "jueza",
    "depositar", "depósito", "orden", "judicial", "mandamiento",
]

IMPORTANT_PATTERNS = {
    "dni": re.compile(r"\b(?:DNI|D\.N\.I\.?|Documento)\s*[:\-]?\s*\d{6,9}\b|\b\d{7,8}\b", re.IGNORECASE),
    "cuil_cuit": re.compile(r"\b(?:CUIL|CUIT|C\.U\.I\.L\.?|C\.U\.I\.T\.?)\s*[:\-]?\s*\d{2}[\s\-.]?\d{7,8}[\s\-.]?\d\b|\b\d{2}[\-\s]?\d{8}[\-\s]?\d\b", re.IGNORECASE),
    "cbu_cvu": re.compile(r"\b(?:CBU|CVU)\s*[:\-]?\s*\d(?:[\s\-.]?\d){15,25}\b", re.IGNORECASE),
    "expediente": re.compile(r"\b(?:expediente|expte|autos|causa)\b.{0,40}\d", re.IGNORECASE),
    "monto": re.compile(r"(?:\$|pesos|ARS|USD)\s*[\d\.]+(?:,\d{2})?|\b\d{1,3}(?:\.\d{3})+(?:,\d{2})?\b", re.IGNORECASE),
    "fecha": re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{1,2}\s+de\s+[a-záéíóúñ]+\s+de\s+\d{4}\b", re.IGNORECASE),
    "alias": re.compile(r"\balias\b\s*[:\-]?\s*[a-z0-9áéíóúñ]+(?:[\.\-][a-z0-9áéíóúñ]+){1,4}", re.IGNORECASE),
    "banco": re.compile(r"\b(?:banco|bancaria|bancario|macro|nacion|nación|galicia|santander|bbva|supervielle|credicoop|brubank|mercado\s*pago)\b", re.IGNORECASE),
    "caratula": re.compile(r"\b(?:caratula|carátula|autos)\b", re.IGNORECASE),
    "juzgado_secretaria": re.compile(r"\b(?:juzgado|secretaria|secretaría|tribunal|camara|cámara)\b", re.IGNORECASE),
}

LEGAL_TERMS_RE = re.compile(r"\b(" + "|".join(re.escape(t) for t in sorted(LEGAL_TERMS, key=len, reverse=True)) + r")\b", re.IGNORECASE)
WORD_RE = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]{2,}")
TOKEN_RE = re.compile(r"\S+")


def safe_text(texto):
    if pd.isna(texto):
        return ""
    return str(texto)


def contar_patrones_importantes(texto):
    texto = safe_text(texto)
    return {nombre: len(patron.findall(texto)) for nombre, patron in IMPORTANT_PATTERNS.items()}


def indicadores_datos_importantes(texto):
    conteos = contar_patrones_importantes(texto)
    return {f"tiene_{k}": v > 0 for k, v in conteos.items()}


def contiene_dato_importante(linea: str) -> bool:
    linea = safe_text(linea)
    if not linea.strip():
        return False
    if LEGAL_TERMS_RE.search(linea):
        return True
    return any(p.search(linea) for p in IMPORTANT_PATTERNS.values())


def medir_calidad_ocr(texto):
    texto = safe_text(texto)
    total = len(texto)
    if total == 0:
        return {
            "len_chars": 0, "len_words": 0, "ratio_alpha": 0.0, "ratio_digits": 0.0,
            "ratio_rare_symbols": 1.0, "ratio_non_printable": 1.0, "line_breaks": 0,
            "one_letter_words": 0, "weird_alnum_tokens": 0, "repeated_char_tokens": 0,
            "near_empty_lines": 0, "symbol_only_lines": 0, "legal_terms_count": 0,
            "normal_word_ratio": 0.0, "important_data_count": 0,
        }

    tokens = TOKEN_RE.findall(texto)
    words = re.findall(r"\b\w+\b", texto, flags=re.UNICODE)
    normal_words = WORD_RE.findall(texto)
    lines = texto.splitlines() or [texto]

    alpha = sum(ch.isalpha() for ch in texto)
    digits = sum(ch.isdigit() for ch in texto)
    punct_ok = set(".,;:()[]{}¿?¡!/-_+$%&@#°ºª'\"\n\r\t ")
    rare_symbols = sum((not ch.isalnum()) and (not ch.isspace()) and ch not in punct_ok for ch in texto)
    non_printable = sum((not ch.isprintable()) and ch not in "\n\r\t" for ch in texto)
    one_letter_words = sum(1 for w in words if len(w) == 1 and w.isalpha())
    weird_alnum = sum(
        1 for tok in tokens
        if re.search(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", tok)
        and re.search(r"\d", tok)
        and not re.search(r"^(?:\d+[/-]\d+|[A-Z]{1,4}\d+|\d+[A-Z]{1,4})$", tok, re.I)
    )
    repeated_char_tokens = sum(1 for tok in tokens if re.search(r"(.)\1{4,}", tok))
    near_empty_lines = sum(1 for line in lines if 0 < len(line.strip()) <= 2)
    symbol_only_lines = sum(1 for line in lines if line.strip() and re.fullmatch(r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+", line.strip()))
    legal_terms_count = len(LEGAL_TERMS_RE.findall(texto))
    important_data_count = sum(v for v in contar_patrones_importantes(texto).values())

    return {
        "len_chars": total,
        "len_words": len(words),
        "ratio_alpha": alpha / total,
        "ratio_digits": digits / total,
        "ratio_rare_symbols": rare_symbols / total,
        "ratio_non_printable": non_printable / total,
        "line_breaks": texto.count("\n"),
        "one_letter_words": one_letter_words,
        "weird_alnum_tokens": weird_alnum,
        "repeated_char_tokens": repeated_char_tokens,
        "near_empty_lines": near_empty_lines,
        "symbol_only_lines": symbol_only_lines,
        "legal_terms_count": legal_terms_count,
        "normal_word_ratio": len(normal_words) / max(len(words), 1),
        "important_data_count": important_data_count,
    }


def calcular_score_ocr(metricas):
    length_score = min(metricas["len_chars"] / 2500, 1.0)
    word_score = min(metricas["len_words"] / 350, 1.0)
    alpha_score = min(metricas["ratio_alpha"] / 0.68, 1.0)
    normal_word_score = min(metricas["normal_word_ratio"] / 0.78, 1.0)
    legal_score = min(metricas["legal_terms_count"] / 12, 1.0)
    data_score = min(metricas["important_data_count"] / 6, 1.0)

    symbol_penalty = min(metricas["ratio_rare_symbols"] / 0.04, 1.0)
    non_printable_penalty = min(metricas["ratio_non_printable"] / 0.01, 1.0)
    one_letter_penalty = min(metricas["one_letter_words"] / max(metricas["len_words"], 1) / 0.20, 1.0)
    weird_token_penalty = min(metricas["weird_alnum_tokens"] / max(metricas["len_words"], 1) / 0.08, 1.0)
    repeated_penalty = min(metricas["repeated_char_tokens"] / max(metricas["len_words"], 1) / 0.05, 1.0)
    symbol_line_penalty = min(metricas["symbol_only_lines"] / max(metricas["line_breaks"] + 1, 1) / 0.20, 1.0)

    positive = (
        0.18 * length_score
        + 0.12 * word_score
        + 0.18 * alpha_score
        + 0.18 * normal_word_score
        + 0.20 * legal_score
        + 0.14 * data_score
    )
    penalty = (
        0.22 * symbol_penalty
        + 0.18 * non_printable_penalty
        + 0.18 * one_letter_penalty
        + 0.18 * weird_token_penalty
        + 0.12 * repeated_penalty
        + 0.12 * symbol_line_penalty
    )
    return float(np.clip(positive * (1 - 0.55 * penalty), 0, 1))


def clasificar_estado_ocr(score, metricas):
    if metricas["len_words"] < 80 or score < 0.22:
        return "irrecuperable"
    if score < 0.42:
        return "muy_ruidoso"
    if score < 0.65:
        return "regular"
    return "legible"

In [6]:
metricas_base = texto_base_tmp.apply(medir_calidad_ocr).apply(pd.Series)
metricas_base["ocr_quality_score"] = metricas_base.apply(lambda row: calcular_score_ocr(row.to_dict()), axis=1)
metricas_base["ocr_estado"] = metricas_base.apply(lambda row: clasificar_estado_ocr(row["ocr_quality_score"], row.to_dict()), axis=1)

print("Distribucion inicial del estado OCR:")
display(metricas_base["ocr_estado"].value_counts().to_frame("cantidad"))

print("Resumen del score inicial:")
display(metricas_base["ocr_quality_score"].describe().to_frame("ocr_quality_score"))

metricas_base.to_csv(REPORT_DIR / "metricas_ocr_base.csv", index=False, encoding="utf-8-sig")

Distribucion inicial del estado OCR:


,cantidad
ocr_estado,
legible,621
regular,4
irrecuperable,1
muy_ruidoso,1


Resumen del score inicial:


,ocr_quality_score
count,627.000000
mean,0.896773
std,0.066111
min,0.202307
25%,0.886506
50%,0.916759
75%,0.932978
max,0.966786


# 4. Limpieza avanzada de OCR

La limpieza es deliberadamente conservadora: normaliza ruido evidente, corrige términos legales cuando el patrón es claro y evita eliminar líneas con datos legales relevantes.

In [7]:
CHAR_TRANSLATION = str.maketrans({
    "\u00a0": " ", "\u2000": " ", "\u2001": " ", "\u2002": " ", "\u2003": " ",
    "\u2004": " ", "\u2005": " ", "\u2006": " ", "\u2007": " ", "\u2008": " ",
    "\u2009": " ", "\u200a": " ", "\u202f": " ", "\u205f": " ", "\u3000": " ",
    "\u200b": "", "\u200c": "", "\u200d": "", "\ufeff": "",
    "“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
    "‘": "'", "’": "'", "‚": "'",
    "–": "-", "—": "-", "−": "-", "‐": "-", "‑": "-",
    "•": "-", "●": "-", "▪": "-", "▫": "-", "◦": "-",
})

OCR_REGEX_REPLACEMENTS = [
    (re.compile(r"\b[Ee][xх][pр][tг]?[eе]?\.?\b"), "expediente"),
    (re.compile(r"\bexped[1il|]ente\b", re.IGNORECASE), "expediente"),
    (re.compile(r"\bexpediente\s*n[ro°º\.]*\b", re.IGNORECASE), "expediente nro."),
    (re.compile(r"\bcar[aá]t[uú]la\b", re.IGNORECASE), "carátula"),
    (re.compile(r"\bcaratula\b", re.IGNORECASE), "carátula"),
    (re.compile(r"\bembarg[o0]\b", re.IGNORECASE), "embargo"),
    (re.compile(r"\bdemandant[e3]\b", re.IGNORECASE), "demandante"),
    (re.compile(r"\bjuzgad[o0]\b", re.IGNORECASE), "juzgado"),
    (re.compile(r"\bsecretar[i1í]a\b", re.IGNORECASE), "secretaría"),
    (re.compile(r"\baut[o0]s\b", re.IGNORECASE), "autos"),
    (re.compile(r"\bbanc[o0]\b", re.IGNORECASE), "banco"),
    (re.compile(r"\bcuent[a4]\b", re.IGNORECASE), "cuenta"),
    (re.compile(r"\btransferenc[i1]a\b", re.IGNORECASE), "transferencia"),
    (re.compile(r"\bc\s*[b8]\s*u\b", re.IGNORECASE), "CBU"),
    (re.compile(r"\bc\s*v\s*u\b", re.IGNORECASE), "CVU"),
    (re.compile(r"\bc\s*u\s*[i1]\s*l\b", re.IGNORECASE), "CUIL"),
    (re.compile(r"\bc\s*u\s*[i1]\s*t\b", re.IGNORECASE), "CUIT"),
    (re.compile(r"\bd\s*n\s*[i1]\b", re.IGNORECASE), "DNI"),
    (re.compile(r"\bm[o0]nt[o0]\b", re.IGNORECASE), "monto"),
    (re.compile(r"\bpes[o0]s\b", re.IGNORECASE), "pesos"),
    (re.compile(r"\bsentenc[i1]a\b", re.IGNORECASE), "sentencia"),
    (re.compile(r"\bmed[i1]da\s+cautelar\b", re.IGNORECASE), "medida cautelar"),
]


def normalizar_basico_ocr(texto):
    texto = safe_text(texto)
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.translate(CHAR_TRANSLATION)
    texto = texto.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")
    texto = "".join(ch if (ch == "\n" or ch == "\t" or not unicodedata.category(ch).startswith("C")) else " " for ch in texto)
    texto = re.sub(r"[ ]+", " ", texto)
    texto = re.sub(r" *\n *", "\n", texto)
    texto = re.sub(r"([,.;:!?])\s*([,.;:!?]){2,}", r"\1", texto)
    texto = re.sub(r"\s+([,.;:!?])", r"\1", texto)
    texto = re.sub(r"([({\[])\s+", r"\1", texto)
    texto = re.sub(r"\s+([)}\]])", r"\1", texto)
    texto = re.sub(r"\n{4,}", "\n\n\n", texto)
    return texto.strip()


def corregir_terminos_ocr(texto):
    # Solo reglas de alta confianza sobre vocabulario legal frecuente.
    for patron, reemplazo in OCR_REGEX_REPLACEMENTS:
        texto = patron.sub(reemplazo, texto)
    return texto


def es_linea_basura_ocr(linea):
    limpia = linea.strip()
    if not limpia:
        return False
    if contiene_dato_importante(limpia):
        return False
    if re.fullmatch(r"[|!¡¿?\-_=~*#.:;,/\\\s]{3,}", limpia):
        return True
    if re.fullmatch(r"(?:[|Il1]\s*){4,}", limpia):
        return True
    if re.fullmatch(r"(?:[_\-~=.]{2,}\s*)+", limpia):
        return True
    if len(limpia) <= 2 and not re.search(r"\d", limpia):
        return True
    if len(limpia) <= 5 and re.fullmatch(r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]*[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]?[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]*", limpia):
        return True
    symbols = sum((not ch.isalnum()) and (not ch.isspace()) for ch in limpia)
    if len(limpia) >= 8 and symbols / max(len(limpia), 1) > 0.75:
        return True
    if re.search(r"(.)\1{8,}", limpia) and not re.search(r"\d{6,}", limpia):
        return True
    return False


def limpiar_linea_ocr(linea):
    if contiene_dato_importante(linea):
        # En lineas con datos sensibles se evita tocar numeros largos. Solo espacios y separadores obvios.
        linea = re.sub(r"[ ]{2,}", " ", linea).strip()
        linea = re.sub(r"\s+([,.;:])", r"\1", linea)
        return linea

    linea = re.sub(r"(?:\|\s*){3,}", " ", linea)
    linea = re.sub(r"[_~]{3,}", " ", linea)
    linea = re.sub(r"-{4,}", " -- ", linea)
    linea = re.sub(r"\.{5,}", " ... ", linea)
    linea = re.sub(r"(.)\1{5,}", r"\1\1\1", linea)
    linea = re.sub(r"[ ]{2,}", " ", linea).strip()
    return linea


def limpiar_ocr_embargo(texto: str) -> str:
    texto = normalizar_basico_ocr(texto)
    texto = corregir_terminos_ocr(texto)

    lineas_limpias = []
    vacia_previa = False
    for linea in texto.split("\n"):
        linea = limpiar_linea_ocr(linea)
        if es_linea_basura_ocr(linea):
            continue
        if not linea.strip():
            if not vacia_previa:
                lineas_limpias.append("")
            vacia_previa = True
            continue
        lineas_limpias.append(linea)
        vacia_previa = False

    texto = "\n".join(lineas_limpias)
    texto = re.sub(r"[ ]{2,}", " ", texto)
    texto = re.sub(r"\n{4,}", "\n\n\n", texto)
    return texto.strip()

# 5. Verificaciones para evitar pérdida excesiva de texto

In [8]:
df_avanzado = df.copy()
df_avanzado["texto_base"] = df_avanzado[texto_col].fillna("").astype(str)
df_avanzado["texto_limpieza_avanzada"] = df_avanzado["texto_base"].apply(limpiar_ocr_embargo)

df_avanzado["len_antes"] = df_avanzado["texto_base"].str.len()
df_avanzado["len_despues"] = df_avanzado["texto_limpieza_avanzada"].str.len()
df_avanzado["chars_eliminados"] = df_avanzado["len_antes"] - df_avanzado["len_despues"]
df_avanzado["porcentaje_reduccion"] = np.where(
    df_avanzado["len_antes"] > 0,
    df_avanzado["chars_eliminados"] / df_avanzado["len_antes"],
    0,
)

df_avanzado["palabras_antes"] = df_avanzado["texto_base"].str.findall(r"\b\w+\b", flags=re.UNICODE).str.len()
df_avanzado["palabras_despues"] = df_avanzado["texto_limpieza_avanzada"].str.findall(r"\b\w+\b", flags=re.UNICODE).str.len()
df_avanzado["porcentaje_reduccion_palabras"] = np.where(
    df_avanzado["palabras_antes"] > 0,
    (df_avanzado["palabras_antes"] - df_avanzado["palabras_despues"]) / df_avanzado["palabras_antes"],
    0,
)

metricas_despues = df_avanzado["texto_limpieza_avanzada"].apply(medir_calidad_ocr).apply(pd.Series)
metricas_despues["ocr_quality_score"] = metricas_despues.apply(lambda row: calcular_score_ocr(row.to_dict()), axis=1)
metricas_despues["ocr_estado"] = metricas_despues.apply(lambda row: clasificar_estado_ocr(row["ocr_quality_score"], row.to_dict()), axis=1)

for col in metricas_despues.columns:
    df_avanzado[col] = metricas_despues[col]

for nombre in IMPORTANT_PATTERNS:
    antes = df_avanzado["texto_base"].apply(lambda t, n=nombre: bool(IMPORTANT_PATTERNS[n].search(safe_text(t))))
    despues = df_avanzado["texto_limpieza_avanzada"].apply(lambda t, n=nombre: bool(IMPORTANT_PATTERNS[n].search(safe_text(t))))
    df_avanzado[f"antes_{nombre}"] = antes
    df_avanzado[f"despues_{nombre}"] = despues
    df_avanzado[f"alerta_desaparece_{nombre}"] = antes & ~despues

alertas_desaparicion = [f"alerta_desaparece_{nombre}" for nombre in IMPORTANT_PATTERNS]
df_avanzado["alerta_reduccion_alta"] = df_avanzado["porcentaje_reduccion"] > 0.35
df_avanzado["alerta_texto_muy_corto"] = df_avanzado["palabras_despues"] < 80
df_avanzado["alerta_posible_perdida_datos"] = df_avanzado[alertas_desaparicion].any(axis=1)

resumen_alertas = pd.DataFrame({
    "metrica": [
        "desaparece_posible_dni",
        "desaparece_posible_cuil_cuit",
        "desaparece_posible_cbu_cvu",
        "desaparece_posible_monto",
        "desaparece_posible_expediente",
        "reduccion_texto_mayor_35",
        "reduccion_palabras_mayor_35",
        "alerta_posible_perdida_datos_total",
    ],
    "cantidad": [
        int(df_avanzado["alerta_desaparece_dni"].sum()),
        int(df_avanzado["alerta_desaparece_cuil_cuit"].sum()),
        int(df_avanzado["alerta_desaparece_cbu_cvu"].sum()),
        int(df_avanzado["alerta_desaparece_monto"].sum()),
        int(df_avanzado["alerta_desaparece_expediente"].sum()),
        int(df_avanzado["alerta_reduccion_alta"].sum()),
        int((df_avanzado["porcentaje_reduccion_palabras"] > 0.35).sum()),
        int(df_avanzado["alerta_posible_perdida_datos"].sum()),
    ],
})

display(resumen_alertas)
resumen_alertas.to_csv(REPORT_DIR / "resumen_alertas_limpieza_avanzada.csv", index=False, encoding="utf-8-sig")

casos_perdida = df_avanzado[df_avanzado["alerta_posible_perdida_datos"]].copy()
print(f"Casos con posible perdida de datos importantes: {len(casos_perdida)}")
if len(casos_perdida) > 0:
    columnas_revision = [c for c in ["id", "nombre", "clasificacion", "ocr_quality_score", "ocr_estado", "porcentaje_reduccion"] if c in casos_perdida.columns]
    display(casos_perdida[columnas_revision + alertas_desaparicion].head(20))

,metrica,cantidad
0,desaparece_posible_dni,0
1,desaparece_posible_cuil_cuit,0
2,desaparece_posible_cbu_cvu,0
3,desaparece_posible_monto,1
4,desaparece_posible_expediente,2
5,reduccion_texto_mayor_35,0
6,reduccion_palabras_mayor_35,0
7,alerta_posible_perdida_datos_total,3


Casos con posible perdida de datos importantes: 3


,id,nombre,clasificacion,ocr_quality_score,ocr_estado,porcentaje_reduccion,alerta_desaparece_dni,alerta_desaparece_cuil_cuit,alerta_desaparece_cbu_cvu,alerta_desaparece_expediente,alerta_desaparece_monto,alerta_desaparece_fecha,alerta_desaparece_alias,alerta_desaparece_banco,alerta_desaparece_caratula,alerta_desaparece_juzgado_secretaria
81,504966,Embargo - usuario,Embargo,0.942496,legible,0.010297,False,False,False,True,False,False,False,False,False,False
344,526787,Embargo - usuario,Embargo,0.876722,legible,0.019652,False,False,False,False,True,False,False,False,False,False
555,543589,Embargo - usuario,Embargo,0.932704,legible,0.001388,False,False,False,True,False,False,False,False,False,False


# 6. Comparación antes/después

In [9]:
def recortar(texto, max_chars=1200):
    texto = safe_text(texto)
    return texto if len(texto) <= max_chars else texto[:max_chars] + "\n[...]"


def diff_textos(antes, despues, max_lines=80):
    antes_lines = recortar(antes, 3000).splitlines()
    despues_lines = recortar(despues, 3000).splitlines()
    diff = list(difflib.unified_diff(antes_lines, despues_lines, fromfile="antes", tofile="despues", lineterm=""))
    if len(diff) > max_lines:
        diff = diff[:max_lines] + ["[...] diff truncado"]
    return "\n".join(diff)


def preparar_comparacion(registros, nombre_reporte):
    filas = []
    for _, row in registros.iterrows():
        filas.append({
            "id": row.get("id", ""),
            "nombre": row.get("nombre", ""),
            "score": row["ocr_quality_score"],
            "estado_ocr": row["ocr_estado"],
            "len_antes": row["len_antes"],
            "len_despues": row["len_despues"],
            "porcentaje_reduccion": row["porcentaje_reduccion"],
            "texto_antes": recortar(row["texto_base"], 1500),
            "texto_despues": recortar(row["texto_limpieza_avanzada"], 1500),
            "diff": diff_textos(row["texto_base"], row["texto_limpieza_avanzada"]),
        })
    reporte = pd.DataFrame(filas)
    reporte.to_csv(REPORT_DIR / f"comparacion_{nombre_reporte}.csv", index=False, encoding="utf-8-sig")
    return reporte


comparaciones = {
    "mejor_score": preparar_comparacion(df_avanzado.sort_values("ocr_quality_score", ascending=False).head(10), "mejor_score"),
    "peor_score": preparar_comparacion(df_avanzado.sort_values("ocr_quality_score", ascending=True).head(10), "peor_score"),
    "mayor_reduccion": preparar_comparacion(df_avanzado.sort_values("porcentaje_reduccion", ascending=False).head(10), "mayor_reduccion"),
    "aleatorios": preparar_comparacion(df_avanzado.sample(min(10, len(df_avanzado)), random_state=RANDOM_STATE), "aleatorios"),
}

for nombre, comp in comparaciones.items():
    print(f"\n=== {nombre.upper()} ===")
    display(comp[["id", "score", "estado_ocr", "len_antes", "len_despues", "porcentaje_reduccion", "texto_antes", "texto_despues"]])


=== MEJOR_SCORE ===


,id,score,estado_ocr,len_antes,len_despues,porcentaje_reduccion,texto_antes,texto_despues
0,542270,0.968725,legible,2703,2698,0.001850,"e\n\nPoder Judicial de la Nación\n\nJUZGADO NACIONAL DE 1RA INSTANCIA DEL\nTRABAJO NRO. 17\n\nOFICIO\n\nBuenos Aires, 6 de Marzo de 2026\nAl Repr Legal de Mercado Libre SRL\nSs/D\n\nTengo el agrado de dirigirme a Ust...","Poder Judicial de la Nación\n\njuzgado NACIONAL DE 1RA INSTANCIA DEL\nTRABAJO NRO. 17\n\nOFICIO\n\nBuenos Aires, 6 de Marzo de 2026\nAl Repr Legal de Mercado Libre SRL\nSs/D\n\nTengo el agrado de dirigirme a Usted en..."
1,536703,0.962489,legible,4423,4418,0.001130,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUN...","(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUN..."
2,542162,0.961814,legible,3410,3407,0.000880,"2)\ne)\n\nJUZGADO NACIONAL DE IRA INSTANCIA DEL TRABAJO\nNRO. 72\n\nPoder Judicial de la Nación\n\nDa\ny 1 a MAR 20%\n""RECEPCION OFICIO.\n\nBuenos Aires, 10 de marzO de 2026.\nAL SEÑOR GERENTE\nMERCADO PAGO SERVICIOS...","2)\n\njuzgado NACIONAL DE IRA INSTANCIA DEL TRABAJO\nNRO. 72\n\nPoder Judicial de la Nación\n\ny 1 a MAR 20%\n""RECEPCION OFICIO.\n\nBuenos Aires, 10 de marzO de 2026.\nAL SEÑOR GERENTE\nMERCADO PAGO SERVICIOS\nDE PRO..."
3,497180,0.957848,legible,5063,5054,0.001778,"0\n\nPoder Judicial de la Nación\n\nJUZGADO CIVIL 45\n\n91273/2013\n\nIncidente N* 1 - ACTOR: BASSO, ALEJANDRA ELENA\ns/BENEFICIO DE LITIGAR SIN GASTOS\n\nOFICIO JUDICIAL\nBuenos Aires, de octubre de 2025\nA Mercado ...","0\n\nPoder Judicial de la Nación\n\njuzgado CIVIL 45\n\n91273/2013\n\nIncidente N* 1 - ACTOR: BASSO, ALEJANDRA ELENA\ns/BENEFICIO DE LITIGAR SIN GASTOS\n\nOFICIO JUDICIAL\nBuenos Aires, de octubre de 2025\nA Mercado ..."
4,536743,0.957797,legible,2959,2946,0.004393,"Poder Judicial de la Nación\n\nJUZGADO COMERCIAL 12\n\nOFICIO\nEMBARGO DE CUENTAS\n\nBuenos Aires , 17 de diciembre de 2025\n\nAl Señor Gerente de Mercado Pago\nAsset Management S.A.\n\nCUIT 33-71862856-9 .\n\n(Merca...","Poder Judicial de la Nación\n\njuzgado COMERCIAL 12\n\nOFICIO\nembargo DE CUENTAS\n\nBuenos Aires, 17 de diciembre de 2025\n\nAl Señor Gerente de Mercado Pago\nAsset Management S.A.\n\nCUIT 33-71862856-9.\n\n(Mercado..."
5,547399,0.957311,legible,2533,2523,0.003948,"20 MAR 2026\n\ngr\nRE ERAS duniciaL REITERATORIO.-\n\nBuenos Aires/, PA marzo de 2026.-\n\nA Mercado Pago.\nS_ // | D.-\n\nTengo el agrado de dirigirme a Ud. en relación a los autos caratulados\n""BCRA (RESOL 124/25) ...","20 MAR 2026\n\nRE ERAS duniciaL REITERATORIO.-\n\nBuenos Aires/, PA marzo de 2026.-\n\nA Mercado Pago.\nS_ // | D.-\n\nTengo el agrado de dirigirme a Ud. en relación a los autos caratulados\n""BCRA (RESOL 124/25) C/ L..."
6,526652,0.957026,legible,4715,4691,0.005090,"26 FER 2006\n\nRECIDEA\n\nS\n\nPoder Judicial de la Nación\nJUZGADO CIVIL 45\n\n91273/2013\n\nIncidente N* 1 - ACTOR: BASSO, ALEJANDRA ELENA\ns/BENEFICIO DE LITIGAR SIN GASTOS\n\nOFICIO JUDICIAL\nBuenos Aires, de oct...","26 FER 2006\n\nRECIDEA\n\nPoder Judicial de la Nación\njuzgado CIVIL 45\n\n91273/2013\n\nIncidente N* 1 - ACTOR: BASSO, ALEJANDRA ELENA\ns/BENEFICIO DE LITIGAR SIN GASTOS\n\nOFICIO JUDICIAL\nBuenos Aires, de octubre ..."
7,536750,0.956522,legible,6571,6573,-0.000304,"OFICIO JUDICIAL\n\nAl Señor Gerente General\nMercado Pago Servicios de Procesamiento SRL\n\nencina Y OS D\n\nTengo el agrado de dirigirme a Ud., en mi carácter de letrado\npatrocinante de la parte actora en autos car...","OFICIO JUDICIAL\n\nAl Señor Gerente General\nMercado Pago Servicios de Procesamiento SRL\n\nencina Y OS D\n\nTengo el agrado de dirigirme a Ud., en mi carácter de letrado\npatrocinante de la parte actora en autos c


=== PEOR_SCORE ===


,id,score,estado_ocr,len_antes,len_despues,porcentaje_reduccion,texto_antes,texto_despues
0,542136,0.220245,irrecuperable,285,258,0.094737,"q a ZA mercad\nA Ó-—\ntArñ20190D 198\n\nPOMADA\n\n40 007. vol sh oben Wisnewslal ssobañb"" Wbiscineua y lorinoo lo\n\nBm 3 me DATE bs D ADA\n\nln MOO .1 7 alansoro.o\ndm y o PrRMOONS 02 |\n\nr ' As\n1\n4 |\na\n1\ny\n!...","q a ZA mercad\nA Ó--\ntArñ20190D 198\n\nPOMADA\n\n40 007. vol sh oben Wisnewslal ssobañb"" Wbiscineua y lorinoo lo\n\nBm 3 me DATE bs D ADA\n\nln MOO.1 7 alansoro.o\ndm y o PrRMOONS 02 |\n\nr ' As\n1\n4 |\n1\nS á.\nba..."
1,543655,0.420600,regular,766,748,0.023499,"l $21200\no o "" Ñ\nA 7 El E : 000 or S ms a y pue hee\nAN ES A\n\n) on\n\ncitado, atendiendo a que es\n\nle la medida cautelar a:\n\n'atricios, CABA de tal\n""conformidad con\n\nconta\ny Y\n\nMa ode Ni 1 em! dos) Por ...","l $21200\no o "" Ñ\nA 7 El E: 000 or S ms a y pue hee\nAN ES A) on\n\ncitado, atendiendo a que es\n\nle la medida cautelar a:\n\n'atricios, CABA de tal\n""conformidad con\n\nconta\ny Y\n\nMa ode Ni 1 em! dos) Por "" a e..."
2,515755,0.530510,regular,749,737,0.016021,"PSN\n| FS\nEl E\n| A o . )\n€ Ai\no. ""z ULA\nPoder Judicial de la Nación |\n| Juzgado Nacional de 1* Instancia del Trabajo N* 48\nl OFICIO JUDICIAL\n¡ Buenos Aires,03 de no de 2026\nAl Sh Director de Asuntos Legales\...","PSN\n| FS\nEl E\n| A o.)\n€ Ai\no. ""z ULA\nPoder Judicial de la Nación |\n| juzgado Nacional de 1* Instancia del Trabajo N* 48\nl OFICIO JUDICIAL\n¡ Buenos Aires,03 de no de 2026\nAl Sh Director de Asuntos Legales\nD..."
3,521002,0.592820,regular,1212,1193,0.015677,"OFICIO JUDICIAL\n\nCapital Federal, a losas del mes de Febrer\n\nente de MERCADO LIBRE SRL\n\nD\n\ntante (conf. art. 61 L.O.), sobre los fondos que la demandada NAZARIA ZAPATOS SA\n0710568975 tenga depositados o que ...","OFICIO JUDICIAL\n\nCapital Federal, a losas del mes de Febrer\n\nente de MERCADO LIBRE SRL\n\ntante (conf. art. 61 L.O.), sobre los fondos que la demandada NAZARIA ZAPATOS SA\n0710568975 tenga depositados o que ingre..."
4,520997,0.597285,regular,861,857,0.004646,"OFICIO DE EMBARGO PREVENTIVO\n\nBuenos Aires, 29 de Diciembré\n\nRCADO PAGO SERVICIOS\nROCESAMIENTO SRL\n/ D\n\nlANDADO: COOPERATIVA DE TRABAJO FENIX LIMITADA Y\n\n». Secretaría única, sito en Sarmiento 1118, 7"" piso...","OFICIO DE embargo PREVENTIVO\n\nBuenos Aires, 29 de Diciembré\n\nRCADO PAGO SERVICIOS\nROCESAMIENTO SRL\n\nlANDADO: COOPERATIVA DE TRABAJO FENIX LIMITADA Y\n\n"". secretaría única, sito en Sarmiento 1118, 7"" piso, de ..."
5,547367,0.619888,regular,706,685,0.029745,"Sy\n\nCERTIFICA que la presente es copia fel cuyo original\nsido firmado digalmente por: ÉRUNELLO, Analia -\n\nPROSECRETARIO/A LETRADO. FASSETTA,\n\nDi Ignacio - JUE,\n\nÑ 5\nÑ 2\n\n. GOACO,\n\nFi di e,\nEM reto guin...","CERTIFICA que la presente es copia fel cuyo original\nsido firmado digalmente por: ÉRUNELLO, Analia -\n\nPROSECRETARIO/A LETRADO. FASSETTA,\n\nDi Ignacio - JUE,\n\nÑ 5\nÑ 2. GOACO,\n\nFi di e,\nEM reto guinana ($\n\n..."
6,508925,0.667858,legible,1249,1248,0.000801,"OFICIODEEMBARGO.\n\nSR|[GERENTE / TITULAR A CARGO\n\nDE MRCADO LIBRE S.R.L..-\nS ooo e cnn D.-\n\ne|dirijo a Ud. en los autos caratulados: ""ORANGE BLUE IMPO «\nEXPORT|TOOLS S.R.L. C/ MAQUINARIAS BOEDO S.R.K. S/ EJECU...","OFICIODEEMBARGO.\n\nSR|[GERENTE / TITULAR A CARGO\n\nDE MRCADO LIBRE S.R.L..-\nS ooo e cnn D.-\n\ne|dirijo a Ud. en los autos caratulados: ""ORANGE BLUE IMPO ""\nEXPORT|TOOLS S.R.L. C/ MAQUINARIAS BOEDO S.R.K. S/ EJECU..."
7,510973,0.690724,legible,995,990,0.005025,"A\nMERCADO LIBRE S.R.L.-\nS 1 D\n\ntrámite ante el Juzgado Nacional de Primera Instancia del Trabajo N"" 35| á\ncargo, Secretaria Única, a cargo del Dr. Gillette Torrilla, Mariano, sito e\n\nDIARTE MANUEL LEONARDO (DN...","MERCADO LIBRE S.R.L.-\nS 1 D\n\ntrámite ante el juzgado Nacional de Primera Instancia del Trabajo N"" 35| á\ncargo, secretaría Única, a cargo del Dr. Gillette Torrilla, Mariano, sito e\n\nDIARTE MANUEL LEONARD


=== MAYOR_REDUCCION ===


,id,score,estado_ocr,len_antes,len_despues,porcentaje_reduccion,texto_antes,texto_despues
0,542136,0.220245,irrecuperable,285,258,0.094737,"q a ZA mercad\nA Ó-—\ntArñ20190D 198\n\nPOMADA\n\n40 007. vol sh oben Wisnewslal ssobañb"" Wbiscineua y lorinoo lo\n\nBm 3 me DATE bs D ADA\n\nln MOO .1 7 alansoro.o\ndm y o PrRMOONS 02 |\n\nr ' As\n1\n4 |\na\n1\ny\n!...","q a ZA mercad\nA Ó--\ntArñ20190D 198\n\nPOMADA\n\n40 007. vol sh oben Wisnewslal ssobañb"" Wbiscineua y lorinoo lo\n\nBm 3 me DATE bs D ADA\n\nln MOO.1 7 alansoro.o\ndm y o PrRMOONS 02 |\n\nr ' As\n1\n4 |\n1\nS á.\nba..."
1,513484,0.852566,legible,2114,1987,0.060076,"]\n\nbdhos transfirió a otro expediente N? 2936//2021-\n\nCEDYLA\n\nideo\n\n'\n\n[0 PODER JUDICIAL DE LA NACION\n[|\nTRIB\n\nDE NOPAFICACIO\nNAL: Juzgado Nacional de Primera Instancia en lo Comercial, 30, des Con\n15...","bdhos transfirió a otro expediente nro.? 2936//2021-\n\nCEDYLA\n\nideo\n\n[0 PODER JUDICIAL DE LA NACION\nTRIB\n\nDE NOPAFICACIO\nNAL: juzgado Nacional de Primera Instancia en lo Comercial, 30, des Con\n1546, Piso 6""..."
2,520982,0.881224,legible,2510,2375,0.053785,"TRIBUNAL: ] a\n\nPeña 1211\n\nFECHA (DE RE\n\nSr: MERCADO |\n\nDOMICILIO:\n\nCARACTER: .\n\nOBSER ACTONE\n\n47\ns.oH. ||\n\nRez:\n\nexpediente\nque tiamita\n\nIntímage a Ñ\n\nordenafla, co;\n$10.00P por\n\nProcesál y...","TRIBUNAL:] a\n\nPeña 1211\n\nFECHA (DE RE\n\nSr: MERCADO |\n\nDOMICILIO:\n\nCARACTER:.\n\nOBSER ACTONE\n\n47\ns.oH. ||\n\nRez:\n\nexpediente\nque tiamita\n\nIntímage a Ñ\n\nordenafla, co;\n$10.00P por\n\nProcesál y 8..."
3,507218,0.880565,legible,1911,1827,0.043956,"| 0 21 FEB 2008\nl\n\nPoder Judicial de la Nación\n\nJUZGADO CIVIL 109\n\nOFICIO\n\nBuenos Aires, 2, de.\n\nparte actora, en tramite ante el JUZGADO DE PRIMERA INTANCIA EN LO\ncargp del la Dra. Puebla, María Belén, s...","| 0 21 FEB 2008\n\nPoder Judicial de la Nación\n\njuzgado CIVIL 109\n\nOFICIO\n\nBuenos Aires, 2, de.\n\nparte actora, en tramite ante el juzgado DE PRIMERA INTANCIA EN LO\ncargp del la Dra. Puebla, María Belén, secr..."
4,525013,0.925660,legible,3177,3082,0.029902,"PODER JUDICIAL DE LA NACION\n\nRECEPCIÓN EN NOTIFICACIONES\n\nSESECRISFARIA No 57 |\nSr: MEREADO HiBRE SRL\n\nDOMIC\n\nCARAC\n\nOBSER\n\nRez:\nii?\n\npi\n\ncumplimiento, daj\n\nIBTO: Caseros 3039, Piso 2 Ciudad Autón...","PODER JUDICIAL DE LA NACION\n\nRECEPCIÓN EN NOTIFICACIONES\n\nSESECRISFARIA No 57 |\nSr: MEREADO HiBRE SRL\n\nDOMIC\n\nCARAC\n\nOBSER\n\nRez:\nii?\n\ncumplimiento, daj\n\nIBTO: Caseros 3039, Piso 2 Ciudad Autónoma de..."
5,547367,0.619888,regular,706,685,0.029745,"Sy\n\nCERTIFICA que la presente es copia fel cuyo original\nsido firmado digalmente por: ÉRUNELLO, Analia -\n\nPROSECRETARIO/A LETRADO. FASSETTA,\n\nDi Ignacio - JUE,\n\nÑ 5\nÑ 2\n\n. GOACO,\n\nFi di e,\nEM reto guin...","CERTIFICA que la presente es copia fel cuyo original\nsido firmado digalmente por: ÉRUNELLO, Analia -\n\nPROSECRETARIO/A LETRADO. FASSETTA,\n\nDi Ignacio - JUE,\n\nÑ 5\nÑ 2. GOACO,\n\nFi di e,\nEM reto guinana ($\n\n..."
6,516905,0.778991,legible,2287,2219,0.029733,"OFICIO.-\n\nBuenos Aires, 10 Febrero de\n|| At. MERCADO PAGO\nASSET MANAGEMENT SA\nS / D\n\nTengo el agrado de dirigirme a Usted,\nautos caratulados ""UMANSKY Adrián Marcelo c/MARQUEZ M:\nAdrián y otro s/EJECUTIVO"", E...","OFICIO.-\n\nBuenos Aires, 10 Febrero de\n|| At. MERCADO PAGO\nASSET MANAGEMENT SA\nS / D\n\nTengo el agrado de dirigirme a Usted,\nautos caratulados ""UMANSKY Adrián Marcelo c/MARQUEZ M:\nAdrián y otro s/EJECUTIVO"", e..."
7,495023,0.890002,legible,1925,1870,0.028571,"OFICIO JUDICIAL\nDE EMBARGO\n\nBA, 11 de diciembre de 2025.\n\n«./ Mercado Pago Asset Management S.A.\nUIT 33-71862856-9)\n\n/ D\n\nE\n\nHa\n\nu\n\nI\no\nto\nD\n\nae\n\nBe\n\nHA\n\nTengo el agrado de dirigirme a uste...","OFICIO JUDICIAL\nDE embargo\n\nBA, 11 de diciembre de 2025.\n\n""./ Mercado Pago Asset Management S.A.\nUIT 33-71862856-9)\n\nTengo el agrado de dirigirme a usted en los autos:\n\nAxpte. 3102/2022 -- ""BARRERA,


=== ALEATORIOS ===


,id,score,estado_ocr,len_antes,len_despues,porcentaje_reduccion,texto_antes,texto_despues
0,544832,0.849330,legible,1698,1688,0.005889,"OFICIO JUDICIAL\n\n/ L/)\n\nBuenos Aires, de Febrero/de/2026\nAl Sr. Apoderado de |\n""MERCADO LIBRE S.R.L.""\nS / D\n\nDe mi consideración:\n\nTengo el agrado de dirigirme a Ud. en los autos\ncaratulados ""CHUQUISACA, ...","OFICIO JUDICIAL\n\nBuenos Aires, de Febrero/de/2026\nAl Sr. Apoderado de |\n""MERCADO LIBRE S.R.L.""\nS / D\n\nDe mi consideración:\n\nTengo el agrado de dirigirme a Ud. en los autos\ncaratulados ""CHUQUISACA, SERGIO HO..."
1,546230,0.952813,legible,5004,4991,0.002598,"A\n\n+ 19 MAR 2026 |\n\n+\n1\nÉ\n\nRECEPCION\n\ne\nPoder Judicial de la Nación\nJUZGADO CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFI...","+ 19 MAR 2026 |\n\n1\n\nRECEPCION\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFICIO REIFERATOR..."
2,543050,0.918374,legible,4379,4337,0.009591,TEXTO Y DATOS DE LA NOTIFICACIÓN\n\n[\n| Usuario conectado:\n\nMINGIACA GABRIEL ALEJANDRO - 202610119308 nofificaciones.scba.gov.ar\n\n| Organismo: JUZGADO DE FAMILIA N? 5 - SAN ISIDRO\n| Carátula: MUGUILLO GERVASIO ...,TEXTO Y DATOS DE LA NOTIFICACIÓN\n\n[| Usuario conectado:\n\nMINGIACA GABRIEL ALEJANDRO - 202610119308 nofificaciones.scba.gov.ar\n\n| Organismo: juzgado DE FAMILIA N? 5 - SAN ISIDRO\n| carátula: MUGUILLO GERVASIO CA...
3,518820,0.859739,legible,3025,3012,0.004298,"a (la\n, CART, . DOCUMENTO\nDESTINATARIO\nPrimera Instancia\nComercial N* 9 MERCADO LIBRE S.R.L.\n13\nDOMICILIO\n1 == >= — yda. € s N* 3039, Piso 2\nEz X= =WX= | LOCALIDAD PROVINGA\npe == =.1= 264A4 Ciudad Autónoma\n...","a (la, CART,. DOCUMENTO\nDESTINATARIO\nPrimera Instancia\nComercial N* 9 MERCADO LIBRE S.R.L.\n13\nDOMICILIO\n1 == >= - yda. € s N* 3039, Piso 2\nEz X= =WX= | LOCALIDAD PROVINGA\npe == =.1= 264A4 Ciudad Autónoma\nY [..."
4,538081,0.947672,legible,5717,5697,0.003498,8 UN 00000\n100009755236\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL N*10 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MA...,8 UN 00000\n100009755236\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\ncarátula DE NOTIFICACION\nOrigen: juzgado EN LO CIVIL Y COMERCIAL N*10 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MA...
5,522288,0.923398,legible,7517,7485,0.004257,"TITZIZO, 9:95 | TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELEGTRÓNICAS\nTEXTO Y DATOS DE LA NOTIFICACIÓN\n\nBURRONI Edgar Jose - 20129293641 Qhnotificaciones.scba.gov.ar\nTRIBUNAL ...","TITZIZO, 9:95 | TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELEGTRÓNICAS\nTEXTO Y DATOS DE LA NOTIFICACIÓN\n\nBURRONI Edgar Jose - 20129293641 Qhnotificaciones.scba.gov.ar\nTRIBUNAL ..."
6,536721,0.903224,legible,5285,5261,0.004541,"""TEXTO Y DATOS DE LA NOTIFICACIÓN\n\nUsuario conectado: SANCHEZ Sebastian Roberto - 202243582030 notificaciones.scba.gov.ar\nOrganismo: JUZGADO EN LO CIVIL Y COMERCIAL N? 4 - SAN NICOLAS\nCarátula: _ NALDO LOMBARDI S...","""TEXTO Y DATOS DE LA NOTIFICACIÓN\n\nUsuario conectado: SANCHEZ Sebastian Roberto - 202243582030 notificaciones.scba.gov.ar\nOrganismo: juzgado EN LO CIVIL Y COMERCIAL N? 4 - SAN NICOLAS\ncarátula: _ NALDO LOMBARDI S..."
7,504422,0.922150,legible,3288,3282,0.001825,"|\n29/12/25, «qm\n\nTEXTO\n\nUsuario cone\nOrgan smo:\n\nCarátula:\nNúmero de c:\nTipo del notifi\nDestinatarios\nFecha notific\n\n| TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELscr...","29/12/25, ""qm\n\nTEXTO\n\nUsuario cone\nOrgan smo:\n\ncarátula:\nNúmero de c:\nTipo del notifi\nDestinatarios\nFecha notific\n\n| TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELscr RÓ..

# 7. Guardado del dataset limpio

In [10]:
columnas_preferidas = [
    "id", "nombre", "clasificacion",
    "texto_ocr", "texto_limpio", "texto_markdown",
    "texto_base", "texto_limpieza_avanzada",
    "len_antes", "len_despues", "chars_eliminados", "porcentaje_reduccion",
    "palabras_antes", "palabras_despues", "porcentaje_reduccion_palabras",
    "ocr_quality_score", "ocr_estado",
    "alerta_reduccion_alta", "alerta_texto_muy_corto", "alerta_posible_perdida_datos",
]
columnas_metricas = [c for c in metricas_despues.columns if c not in columnas_preferidas]
columnas_alertas = [c for c in df_avanzado.columns if c.startswith("alerta_desaparece_")]
columnas_indicadores = [c for c in df_avanzado.columns if c.startswith("antes_") or c.startswith("despues_")]
columnas_originales_extra = [c for c in df_avanzado.columns if c not in columnas_preferidas + columnas_metricas + columnas_alertas + columnas_indicadores]
columnas_salida = [c for c in columnas_preferidas if c in df_avanzado.columns] + columnas_metricas + columnas_alertas + columnas_indicadores + columnas_originales_extra

# Se guarda el dataset completo sin borrar registros ni sobrescribir embargos_limpios.csv.
df_avanzado[columnas_salida].to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Dataset completo guardado en: {OUTPUT_PATH}")
print("Filas guardadas:", len(df_avanzado))
print("Columnas guardadas:", len(columnas_salida))

Dataset completo guardado en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_limpieza_avanzada.csv
Filas guardadas: 627
Columnas guardadas: 73


# 8. Selección de los 15 mejores embargos

In [11]:
for nombre in IMPORTANT_PATTERNS:
    df_avanzado[f"tiene_{nombre}"] = df_avanzado["texto_limpieza_avanzada"].apply(lambda t, n=nombre: bool(IMPORTANT_PATTERNS[n].search(safe_text(t))))

indicadores_top = [f"tiene_{nombre}" for nombre in IMPORTANT_PATTERNS]
df_avanzado["cantidad_indicadores_importantes"] = df_avanzado[indicadores_top].sum(axis=1)
df_avanzado["score_seleccion_gliner"] = (
    df_avanzado["ocr_quality_score"] * 0.55
    + np.minimum(df_avanzado["palabras_despues"] / 700, 1.0) * 0.15
    + np.minimum(df_avanzado["cantidad_indicadores_importantes"] / 8, 1.0) * 0.20
    + (1 - df_avanzado["porcentaje_reduccion"].clip(0, 1)) * 0.10
)

elegibles = df_avanzado[
    (df_avanzado["ocr_estado"] == "legible")
    & (df_avanzado["palabras_despues"] >= 120)
    & (df_avanzado["porcentaje_reduccion"] <= 0.35)
    & (~df_avanzado["alerta_posible_perdida_datos"])
].copy()

if len(elegibles) < 15:
    print(f"Solo hay {len(elegibles)} registros legibles estrictos. Se completa el ranking con regulares sin alertas graves.")
    elegibles = df_avanzado[
        (df_avanzado["ocr_estado"].isin(["legible", "regular"]))
        & (df_avanzado["palabras_despues"] >= 120)
        & (df_avanzado["porcentaje_reduccion"] <= 0.35)
        & (~df_avanzado["alerta_posible_perdida_datos"])
    ].copy()

top15 = elegibles.sort_values(
    ["score_seleccion_gliner", "ocr_quality_score", "cantidad_indicadores_importantes", "palabras_despues"],
    ascending=False,
).head(15).copy()

top15["texto_limpio_avanzado_preview_500"] = top15["texto_limpieza_avanzada"].str.slice(0, 500)

columnas_top15 = [
    "id", "nombre", "clasificacion", "ocr_quality_score", "score_seleccion_gliner", "ocr_estado",
    "len_despues", "palabras_despues", "porcentaje_reduccion", "cantidad_indicadores_importantes",
] + indicadores_top + ["texto_limpieza_avanzada", "texto_limpio_avanzado_preview_500"]
columnas_top15 = [c for c in columnas_top15 if c in top15.columns]

top15[columnas_top15].to_csv(TOP15_PATH, index=False, encoding="utf-8-sig")

print(f"Top 15 guardado en: {TOP15_PATH}")
print("Registros seleccionados:", len(top15))

display(top15[[c for c in ["id", "ocr_quality_score", "score_seleccion_gliner", "ocr_estado", "len_despues", "porcentaje_reduccion", "texto_limpio_avanzado_preview_500"] if c in top15.columns]])

Top 15 guardado en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_15_mejores_legibles.csv
Registros seleccionados: 15


,id,ocr_quality_score,score_seleccion_gliner,ocr_estado,len_despues,porcentaje_reduccion,texto_limpio_avanzado_preview_500
444,536703,0.962489,0.979256,legible,4418,0.001130,"(Ue\n\n413/26, 09-18 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA- NOTIFICACIONES ELECTRÓNICAS\nUsuario conectado: DOS SANTOS CARLOS ROBERTO - 23137661349Q notificaciones.scba.govar\nOrganismo: TRIBUN..."
511,542140,0.955085,0.975095,legible,4447,0.002020,"Poder Judicial de la Nación\njuzgado CIVIL 92\n\n29921/2018\n\nIncidente N* 1 - ACTOR: GONZALES\nGONZALES, JUANA JAQUELINE Y OTRO\nDEMANDADO: GUTIERREZ MONTES, ELOY\nES mercadO E ECUCION DE ALIMENTOS - INCIDENTE\nlib..."
589,546228,0.953850,0.974349,legible,4826,0.002687,"a 1+\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\n\ns/ALIMENTOS\npoa Buenos Aires, de marzo de 2026.- MEG\n\nTy Z OFICIO REITERATORIO\n\nMercado Pago..."
391,531044,0.953356,0.974188,legible,5060,0.001579,"18/2/26, 10:49 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS dl 2)\n\nscba.gov.al\n\n7 Presentaciones y\n\nÑ Notificaciones Electrónicas\nA SUPREMA CORTE DE JUSTICIA\n\n<<..."
590,546230,0.952813,0.973788,legible,4991,0.002598,"+ 19 MAR 2026 |\n\n1\n\nRECEPCION\n\nPoder Judicial de la Nación\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG\nOFICIO REIFERATOR..."
591,546231,0.953337,0.973750,legible,4754,0.005855,10)\n\n19 MAR 2026\n\nOFICIO\nBuenos Aires 23 de diciembre de 2025.\n\nAl Señor Gerente de Mercado Pago Asset Management S.A.\nCurr 33-71862856-9. e\n\n4\n\nTengo el agrado de dirigirme a Usted en los autos\ncaratula...
465,536741,0.952666,0.973564,legible,5445,0.004024,"OFICIO\n\nembargo DE CUENTAS\n\n1.0 MAR 206\n\nBuenos Aires, de diciembre de 2025\nRECIBIDO\nAl Señor Gerente de Mercado Pago Asset Management S.A.\nCUIT 33-71862856-9. (Mercado Fondos)\n\nSs / D\n\nTengo el agrado d..."
610,547358,0.951686,0.973308,legible,5017,0.001195,"19/3/26, 17:47. about:blank\nDATOS NOTIFICACION ELECTRONICA\nUsuario conectado: MARESCO Graciela Silvia\n\nOrganismo: juzgado DE FAMILIA N? 5 - AVELLANEDA\ncarátula: MARESCO GRACIELA SILVIA C/RIOBO SEBASTIAN LEANDRO ..."
371,529790,0.951999,0.973271,legible,6379,0.003281,"0.3 MAR 2006 OFICIO\n\nRECIBIDO\n\nSan Martin, Y. de: YIALTO de2026.-\n\ndar do\n\nAl Sr. Gerente del\nMercado Libre S.R.L. E\n\nPresente.-\n\nTengo el agrado de dirigirme a Ud. en los autos caratulados:\n""SINDICATO ..."
598,546250,0.950877,0.972046,legible,4971,0.009366,"cado\n\nUNS\npus is,\n\n2 MAR 2026\nRECEPCION\n\n1\n\nPoder Judicial de la Nación\n\njuzgado CIVIL 7\n\n40471/2023\n\nVASSALLO, FERNANDO c/ BENITEZ, MARIA ANGELICA\ns/ALIMENTOS\n\nBuenos Aires, de marzo de 2026.- MEG..."


# 9. Reporte final

La siguiente celda genera un resumen reproducible del proceso y lo guarda en `data/processed/reportes_embargos/resumen_final_limpieza_avanzada_embargos.json`.

In [12]:
conteo_estados = df_avanzado["ocr_estado"].value_counts().to_dict()
resumen_final = {
    "total_embargos_procesados": int(len(df_avanzado)),
    "textos_legibles": int(conteo_estados.get("legible", 0)),
    "textos_regulares": int(conteo_estados.get("regular", 0)),
    "textos_muy_ruidosos": int(conteo_estados.get("muy_ruidoso", 0)),
    "textos_irrecuperables": int(conteo_estados.get("irrecuperable", 0)),
    "porcentaje_promedio_reduccion_texto": round(float(df_avanzado["porcentaje_reduccion"].mean() * 100), 2),
    "alertas_reduccion_alta": int(df_avanzado["alerta_reduccion_alta"].sum()),
    "alertas_posible_perdida_datos": int(df_avanzado["alerta_posible_perdida_datos"].sum()),
    "registros_seleccionados_etapa_posterior": int(len(top15)),
    "archivo_dataset_completo": str(OUTPUT_PATH),
    "archivo_15_mejores": str(TOP15_PATH),
    "carpeta_reportes": str(REPORT_DIR),
    "proximos_pasos": "Validar manualmente los 15 mejores embargos y luego, en una etapa separada, probar GLiNER para extraccion de entidades.",
}

with open(REPORT_DIR / "resumen_final_limpieza_avanzada_embargos.json", "w", encoding="utf-8") as f:
    json.dump(resumen_final, f, ensure_ascii=False, indent=2)

resumen_markdown = f"""
## Resumen final del proceso

- Cantidad total de embargos procesados: {resumen_final['total_embargos_procesados']}
- Textos legibles: {resumen_final['textos_legibles']}
- Textos regulares: {resumen_final['textos_regulares']}
- Textos muy ruidosos: {resumen_final['textos_muy_ruidosos']}
- Textos irrecuperables: {resumen_final['textos_irrecuperables']}
- Porcentaje promedio de reduccion de texto: {resumen_final['porcentaje_promedio_reduccion_texto']}%
- Alertas por reduccion alta: {resumen_final['alertas_reduccion_alta']}
- Alertas por posible perdida de datos: {resumen_final['alertas_posible_perdida_datos']}
- Registros seleccionados para etapa posterior: {resumen_final['registros_seleccionados_etapa_posterior']}
- Dataset completo generado: `{OUTPUT_PATH}`
- Seleccion de 15 mejores embargos: `{TOP15_PATH}`
- Reportes de verificacion: `{REPORT_DIR}`

Proximo paso sugerido: validar manualmente los 15 mejores embargos. Despues de esa validacion, realizar una etapa separada para probar modelos GLiNER y extraer entidades.
"""

try:
    from IPython.display import Markdown, display
    display(Markdown(resumen_markdown))
except Exception:
    print(resumen_markdown)


## Resumen final del proceso

- Cantidad total de embargos procesados: 627
- Textos legibles: 621
- Textos regulares: 5
- Textos muy ruidosos: 0
- Textos irrecuperables: 1
- Porcentaje promedio de reduccion de texto: 0.58%
- Alertas por reduccion alta: 0
- Alertas por posible perdida de datos: 3
- Registros seleccionados para etapa posterior: 15
- Dataset completo generado: `E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_limpieza_avanzada.csv`
- Seleccion de 15 mejores embargos: `E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_15_mejores_legibles.csv`
- Reportes de verificacion: `E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\reportes_embargos`

Proximo paso sugerido: validar manualmente los 15 mejores embargos. Despues de esa validacion, realizar una etapa separada para probar modelos GLiNER y extraer entidades.
